# ATML PA3 (Fall 2026) – Task 1: LLM Fine-Tuning Colab Workflow

This interactive notebook executes **Task 1: Large Language Model Fine-Tuning** on a **Tesla T4 GPU (16 GiB VRAM)** with full **Google Drive persistence**.

### Experimental Conditions
1. **Full Fine-Tuning (Full FT)**: LR `2e-5`, $\lambda = 0$
2. **Full Fine-Tuning with L2-SP**: LR `2e-5`, $\lambda = 10^{-3}$, proximity regularization toward $\theta_0$
3. **LoRA Rank 4**: $(r, \alpha) = (4, 8)$, LR `2e-4`, all 7 projection modules in 28 layers
4. **LoRA Rank 16**: $(r, \alpha) = (16, 32)$, LR `2e-4`
5. **OPLoRA Rank 16**: $(r, \alpha) = (16, 32)$, protection rank $k=128$, leading singular basis projection

### Target Budget
- 40,000 eligible MetaMathQA examples ($\le 2048$ tokens, GSM8K test leak filtered)
- 1,250 optimizer updates (microbatch 2, accum 16, effective batch 32)


## 1. Mount Google Drive for Persistent Storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSISTENT_ROOT = '/content/drive/MyDrive/ATML_PA3_Task1'
!mkdir -p {PERSISTENT_ROOT}/checkpoints
!mkdir -p {PERSISTENT_ROOT}/artifacts
!mkdir -p {PERSISTENT_ROOT}/results
!mkdir -p {PERSISTENT_ROOT}/logs
print(f'Persistent root verified at {PERSISTENT_ROOT}')

## 2. Clone Repository & Setup Persistent Symlinks

In [ ]:
%cd /content
# Set your repository URL here:
REPO_URL = 'https://github.com/Saad-0000/atml-pa3-FineTuning.git'
![ -d repo ] && (cd repo && git pull origin master) || git clone {REPO_URL} repo
%cd /content/repo/task1

# Replace local ephemeral directories with persistent Drive symlinks
!rm -rf checkpoints artifacts results logs
!ln -s {PERSISTENT_ROOT}/checkpoints checkpoints
!ln -s {PERSISTENT_ROOT}/artifacts artifacts
!ln -s {PERSISTENT_ROOT}/results results
!ln -s {PERSISTENT_ROOT}/logs logs
!mkdir -p /content/repo/shared_assets/inputs
!cp {PERSISTENT_ROOT}/shared_assets/inputs/first_plot_questions.yaml /content/repo/shared_assets/inputs/ 2>/dev/null || true

import os
os.environ['PYTHONPATH'] = f'/content/repo/task1/src:{os.environ.get("PYTHONPATH", "")}'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Persistent symlinks established.')

## 3. Verify Hardware & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -r requirements-lock.txt

## 4. Preflight & Smoke Test

In [ ]:
!python scripts/preflight.py --persistent-root {PERSISTENT_ROOT}

## 5. Prepare Data (40k MetaMathQA + 64 Activation Probes)

In [ ]:
!python scripts/prepare_data.py

## 6. Baseline Evaluation & Reference Asset Caching

In [ ]:
!python scripts/evaluate_baseline.py

## 7. Train Method 1: Full Fine-Tuning (Full FT)

In [ ]:
!python scripts/train_method.py --method full_ft --resume auto

## 8. Train Method 2: Full Fine-Tuning + L2-SP

In [ ]:
!python scripts/train_method.py --method l2sp --resume auto

## 9. Train Method 3: LoRA Rank 4

In [ ]:
!python scripts/train_method.py --method lora4 --resume auto

## 10. Train Method 4: LoRA Rank 16

In [ ]:
!python scripts/train_method.py --method lora16 --resume auto

## 11. Train Method 5: OPLoRA Rank 16 (Protection Rank 128)

In [ ]:
!python scripts/train_method.py --method oplora16 --resume auto

## 12. Evaluate Required Conditions Across 4 Benchmarks

In [ ]:
!python scripts/evaluate_required.py --condition all

## 13. Weight-Space & Activation Diagnostics (Layers 0, 13, 27)

In [ ]:
!python scripts/run_diagnostics.py

## 14. Qualitative Evaluation (168 Responses & 42-Annotation Sheet)

In [ ]:
!python scripts/run_qualitative.py

## 15. Build Authoritative Results (Tables & Plots)

In [ ]:
!python scripts/build_results.py

## 16. Evidence Completeness Audit

In [ ]:
!python scripts/audit_evidence.py